# 🛒 Retail Sales — End-to-End ML Project
### Phases 1 → 4: Cleaning · EDA · Modelling · Hyperparameter Tuning
**Dataset:** UCI Retail Sales (274 orders, 13 features, Jan–Oct 2024)**  
**Task:** Binary classification — predict high customer rating (≥ 4 stars)**  
**Author:** Generated by automated pipeline  


## 0. Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
matplotlib.use("Agg")

from sklearn.model_selection import (
    train_test_split, GridSearchCV, RandomizedSearchCV,
    StratifiedKFold, learning_curve, cross_val_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve,
    precision_score, recall_score, f1_score
)
from sklearn.preprocessing import StandardScaler
from scipy.stats import randint, uniform

print("✅ All imports successful")


---
## Phase 1 — Data Cleaning
> Remove duplicates · fill missing values · cap outliers · engineer revenue

In [ ]:
DATA_PATH = "cleaned_dataset.csv"

df_raw = pd.read_csv(DATA_PATH)
print(f"Raw shape: {df_raw.shape}")
df_raw.head(3)


In [ ]:
# 1. Drop duplicates
df = df_raw.drop_duplicates()
print(f"After dedup: {len(df)} rows")

# 2. Fill missing values with median
NUM_COLS = ["quantity", "unit_price", "discount_pct", "customer_age", "rating"]
for col in NUM_COLS:
    n = df[col].isna().sum()
    if n: df[col] = df[col].fillna(df[col].median())
print(f"Nulls remaining: {df.isna().sum().sum()}")

# 3. Cap outliers using IQR
for col in NUM_COLS:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    df[col] = df[col].clip(q1 - 1.5*iqr, q3 + 1.5*iqr)

# 4. Recompute revenue
df["revenue"] = (df["quantity"] * df["unit_price"] * (1 - df["discount_pct"]/100)).round(2)
print(f"Revenue range: ₹{df.revenue.min():.0f} – ₹{df.revenue.max():.0f}")
df.describe().round(2)


---
## Phase 2 — Exploratory Data Analysis
> Distributions · correlations · group aggregations

In [ ]:
# Class balance
df["high_rating"] = (df["rating"] >= 4.0).astype(int)
print("Class balance:")
print(df["high_rating"].value_counts(normalize=True).mul(100).round(1))


In [ ]:
# Revenue by category
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
df.groupby("category")["revenue"].sum().sort_values().plot(
    kind="barh", ax=axes[0], color="#2ecc71", title="Revenue by Category")
df.groupby("region")["revenue"].sum().sort_values().plot(
    kind="barh", ax=axes[1], color="#3498db", title="Revenue by Region")
df["rating"].value_counts().sort_index().plot(
    kind="bar", ax=axes[2], color="#e67e22", title="Rating Distribution", rot=0)
plt.tight_layout()
plt.savefig("eda_summary.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved: eda_summary.png")


In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(8, 5))
num_df = df[["quantity","unit_price","discount_pct","customer_age","revenue","rating"]]
sns.heatmap(num_df.corr().round(2), annot=True, fmt=".2f",
            cmap="YlGn", ax=ax, linewidths=0.5)
ax.set_title("Correlation Matrix", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("correlation_matrix.png", dpi=130, bbox_inches="tight")
plt.show()


---
## Phase 3 — Baseline ML Models
> Logistic Regression · Decision Tree · Random Forest

**Why these three?**
- Logistic Regression: simple, interpretable, good baseline
- Decision Tree: non-linear but overfit-prone — useful to compare
- Random Forest: ensemble of trees, usually the strongest of the three

In [ ]:
# Feature engineering & encoding
df["revenue_per_unit"] = df["revenue"] / df["quantity"].replace(0, float("nan"))
df["has_discount"] = (df["discount_pct"] > 0).astype(int)

for col in ["category", "region", "payment_method"]:
    df[col + "_enc"] = df[col].astype("category").cat.codes

FEATURES = [
    "quantity", "unit_price", "discount_pct", "customer_age",
    "month", "revenue", "revenue_per_unit", "has_discount",
    "category_enc", "region_enc", "payment_method_enc",
]
X = df[FEATURES].fillna(df[FEATURES].median())
y = df["high_rating"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Train: {X_train.shape}  Test: {X_test.shape}")
print(f"Positive class (y=1): {y_train.mean()*100:.1f}% of train")


In [ ]:
# Train & compare baseline models
models_base = {
    "Logistic Regression":  LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "Decision Tree":        DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=42),
    "Random Forest":        RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42),
}

baseline_results = {}
for name, model in models_base.items():
    Xtr = X_train_sc if "Logistic" in name else X_train
    Xte = X_test_sc  if "Logistic" in name else X_test
    model.fit(Xtr, y_train)
    y_pred  = model.predict(Xte)
    y_proba = model.predict_proba(Xte)[:,1]
    baseline_results[name] = {
        "acc": accuracy_score(y_test, y_pred)*100,
        "auc": roc_auc_score(y_test, y_proba),
    }
    print(f"{name:<25} Acc={baseline_results[name]["acc"]:.1f}%  AUC={baseline_results[name]["auc"]:.3f}")


---
## Phase 4 — Hyperparameter Tuning

### Concept: What is a hyperparameter?
> Regular parameters (weights) are **learned** during training.  
> Hyperparameters are settings you **choose before training** — e.g. how deep a tree can grow, how strong the regularisation is.  
> Tuning finds the combination that maximises your validation metric.

### Two main strategies
| Strategy | How it works | When to use |
|---|---|---|
| **GridSearchCV** | Try every combination in a predefined grid | Small grids (< 200 combos) |
| **RandomizedSearchCV** | Sample n_iter random combos from distributions | Large grids, continuous params |


### 4a. GridSearchCV — Logistic Regression
> Finding the best regularisation strength (C) and penalty type (L1 vs L2)

In [ ]:
param_grid_lr = {
    "C":       [0.001, 0.01, 0.1, 1, 10, 100],
    "penalty": ["l1", "l2"],
    "solver":  ["liblinear"],
}
# C controls regularisation strength:
#   Small C  → strong regularisation → simpler model → less overfit
#   Large C  → weak  regularisation → complex model → may overfit

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_lr = GridSearchCV(
    LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    param_grid_lr,
    cv=cv_strategy,
    scoring="roc_auc",   # ← we optimise AUC, not raw accuracy
    n_jobs=-1,
)
grid_lr.fit(X_train_sc, y_train)

print("Best params:", grid_lr.best_params_)
print(f"Best CV AUC: {grid_lr.best_score_:.4f}")
lr_tuned = grid_lr.best_estimator_


In [ ]:
# Visualise the grid: C vs AUC for each penalty
results_df = pd.DataFrame(grid_lr.cv_results_)
pivot = results_df.pivot_table(
    index="param_penalty", columns="param_C", values="mean_test_score"
).astype(float)

fig, ax = plt.subplots(figsize=(9, 3))
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="YlGn", ax=ax,
            linewidths=0.5, cbar_kws={"label": "Mean AUC"})
ax.set_title("GridSearch: Logistic Regression — AUC by C & Penalty",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("gridsearch_heatmap.png", dpi=130, bbox_inches="tight")
plt.show()


### 4b. RandomizedSearchCV — Random Forest
> Sampling 40 random combinations from a large hyperparameter space

In [ ]:
param_dist_rf = {
    "n_estimators":      randint(50, 500),        # how many trees
    "max_depth":         [None, 3, 5, 8, 12, 20], # max depth per tree
    "min_samples_split": randint(2, 20),          # node must have >= N samples to split
    "min_samples_leaf":  randint(1, 10),          # leaf must have >= N samples
    "max_features":      ["sqrt", "log2", None],  # features considered at each split
    "class_weight":      ["balanced", "balanced_subsample"],
}

rscv_rf = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=param_dist_rf,
    n_iter=40,
    cv=cv_strategy,
    scoring="roc_auc",
    n_jobs=-1,
    random_state=42,
)
rscv_rf.fit(X_train, y_train)

print("Best params:", rscv_rf.best_params_)
print(f"Best CV AUC: {rscv_rf.best_score_:.4f}")
rf_tuned = rscv_rf.best_estimator_


### 4c. Gradient Boosting (bonus model)
> Trees built sequentially — each tree corrects the errors of the previous one

In [ ]:
param_dist_gb = {
    "n_estimators":     randint(50, 300),
    "max_depth":        randint(2, 6),
    "learning_rate":    uniform(0.01, 0.3),  # step size — lower = slower but more precise
    "subsample":        uniform(0.6, 0.4),   # fraction of samples used per tree
    "min_samples_leaf": randint(1, 15),
}

rscv_gb = RandomizedSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_distributions=param_dist_gb,
    n_iter=30,
    cv=cv_strategy,
    scoring="roc_auc",
    n_jobs=-1,
    random_state=42,
)
rscv_gb.fit(X_train, y_train)
print("Best params:", rscv_gb.best_params_)
print(f"Best CV AUC: {rscv_gb.best_score_:.4f}")
gb_tuned = rscv_gb.best_estimator_


### 4d. Evaluate all tuned models on the held-out test set

In [ ]:
tuned_models = {
    "LR Tuned":  (lr_tuned, X_test_sc),
    "RF Tuned":  (rf_tuned, X_test),
    "GB Tuned":  (gb_tuned, X_test),
}

tuned_results = {}
for name, (model, Xte) in tuned_models.items():
    y_pred  = model.predict(Xte)
    y_proba = model.predict_proba(Xte)[:,1]
    tuned_results[name] = {
        "acc":  accuracy_score(y_test, y_pred)*100,
        "auc":  roc_auc_score(y_test, y_proba),
        "f1":   f1_score(y_test, y_pred),
    }
    print(f"{name:<15}  Acc={tuned_results[name]["acc"]:.1f}%  "
          f"AUC={tuned_results[name]["auc"]:.3f}  "
          f"F1={tuned_results[name]["f1"]:.3f}")


### 4e. Learning Curves — diagnose bias vs variance
> **Train score >> Val score** → overfitting (high variance) → need more data or regularisation  
> **Both scores low** → underfitting (high bias) → need a more complex model

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, model, Xtr, color) in zip(axes, [
    ("LR Tuned",  lr_tuned, X_train_sc, "#2d6a4f"),
    ("RF Tuned",  rf_tuned, X_train,    "#3498db"),
]):
    ts, tr_sc, val_sc = learning_curve(
        model, Xtr, y_train,
        train_sizes=np.linspace(0.1, 1.0, 8),
        cv=cv_strategy, scoring="roc_auc", n_jobs=-1
    )
    ax.plot(ts, tr_sc.mean(1),  color=color,     lw=2, label="Train AUC")
    ax.fill_between(ts, tr_sc.mean(1)-tr_sc.std(1),
                        tr_sc.mean(1)+tr_sc.std(1), alpha=0.15, color=color)
    ax.plot(ts, val_sc.mean(1), color="#e67e22", lw=2, label="Val AUC")
    ax.fill_between(ts, val_sc.mean(1)-val_sc.std(1),
                        val_sc.mean(1)+val_sc.std(1), alpha=0.15, color="#e67e22")
    ax.set_title(f"Learning Curve — {name}", fontsize=11, fontweight="bold")
    ax.set_xlabel("Training Samples")
    ax.set_ylabel("AUC")
    ax.legend(fontsize=9)
    ax.spines[["top","right"]].set_visible(False)

plt.tight_layout()
plt.savefig("learning_curves.png", dpi=130, bbox_inches="tight")
plt.show()


### 4f. Threshold Tuning — Precision / Recall trade-off
> The default threshold (0.5) is not always optimal.  
> Lowering it → catch more positives (higher recall)  
> Raising it → fewer false alarms (higher precision)

In [ ]:
# Use best model by AUC for threshold analysis
best_name = max(tuned_results, key=lambda n: tuned_results[n]["auc"])
best_model = tuned_models[best_name][0]
best_Xte   = tuned_models[best_name][1]
y_proba_best = best_model.predict_proba(best_Xte)[:,1]

thresholds = np.arange(0.1, 0.91, 0.05)
rows = []
for t in thresholds:
    yp = (y_proba_best >= t).astype(int)
    rows.append({"threshold": t,
                 "precision": precision_score(y_test, yp, zero_division=0),
                 "recall":    recall_score(y_test, yp, zero_division=0),
                 "f1":        f1_score(y_test, yp, zero_division=0)})
thresh_df = pd.DataFrame(rows)

best_t = thresh_df.loc[thresh_df["f1"].idxmax(), "threshold"]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresh_df["threshold"], thresh_df["precision"], "g-o", ms=4, label="Precision")
ax.plot(thresh_df["threshold"], thresh_df["recall"],    "r-s", ms=4, label="Recall")
ax.plot(thresh_df["threshold"], thresh_df["f1"],        "b-^", ms=4, label="F1 Score")
ax.axvline(best_t, color="red",  lw=2, ls="--", label=f"Best F1 @ {best_t:.2f}")
ax.axvline(0.5,    color="gray", lw=1, ls=":",  label="Default (0.50)")
ax.set_title(f"Threshold Tuning — {best_name}", fontsize=12, fontweight="bold")
ax.set_xlabel("Decision Threshold")
ax.set_ylabel("Score")
ax.legend()
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig("threshold_tuning.png", dpi=130, bbox_inches="tight")
plt.show()
print(f"Optimal F1 threshold: {best_t:.2f}")


### 4g. Final comparison: Baseline vs Tuned

In [ ]:
comparison = []
for name, res in baseline_results.items():
    comparison.append({"Model": name + " (baseline)", "Accuracy": res["acc"], "AUC": res["auc"]})
for name, res in tuned_results.items():
    comparison.append({"Model": name, "Accuracy": res["acc"], "AUC": res["auc"]})

comp_df = pd.DataFrame(comparison).sort_values("AUC", ascending=False)
comp_df[["Accuracy", "AUC"]] = comp_df[["Accuracy", "AUC"]].round(3)
print(comp_df.to_string(index=False))


---
## Summary & Next Steps

| Phase | What was done | Key output |
|---|---|---|
| 1 | Cleaning: dedup, impute, IQR capping |  |
| 2 | EDA: distributions, correlations, group stats |  |
| 3 | Baseline models: LR, DT, RF |  |
| 4 | Tuning: GridSearch, RandomSearch, LR curves, thresholds |  |

### Suggested Phase 5 options
- **XGBoost / LightGBM** — faster, usually stronger gradient boosting
- **SHAP values** — explain *why* the model made each prediction
- **SMOTE** — synthetic oversampling for class imbalance
- **FastAPI deployment** — serve predictions over an HTTP endpoint
- **NLP on product descriptions** — if you add a text column
